In [40]:
import sys
from pathlib import Path

# Define the path to the directory containing compute_rms.py
path_to_compute_rms = Path("../errpca_pt_compute_rms")

# Add this path to sys.path
sys.path.append(str(path_to_compute_rms))


In [71]:
import numpy as np
from numpy.linalg import slogdet
from scipy.sparse import issparse, csr_matrix
# from compute_rms import compute_rms  # Ensure this module is available or replace with appropriate code

def cf_full(X, A, S, Mu, V, Av=None, Sv=None, Isv=None, Muv=None, Va=None, Vmu=None, M=None, sXv=None, ndata=None):
    n1, n2 = X.shape
    ncomp = A.shape[1]

    # Handle M, sXv, ndata if not provided
    if M is None:
        M = ~np.isnan(X)
        X = np.where(M, X, 0)

    if sXv is None:
        IX, JX = np.where(M)
        ndata = len(IX)

        # Since compute_rms is not provided, we'll compute rms as an example
        X_hat = A @ S + Mu[:, np.newaxis]
        diff = (X - X_hat) * M
        rms = np.sqrt(np.sum(diff ** 2) / ndata)

        # Initialize sXv based on rms
        sXv = (rms ** 2) * ndata
        if Sv is None:
            for r in range(ndata):
                i = IX[r]
                j = JX[r]
                a_i = A[i, :]
                sv_j = np.eye(ncomp)
                sXv += a_i @ sv_j @ a_i.T
                if Av is not None and len(Av) > i:
                    s_j = S[:, j]
                    av_i = Av[i]
                    sXv += s_j.T @ av_i @ s_j + np.sum(sv_j * av_i)
        else:
            for r in range(ndata):
                i = IX[r]
                j = JX[r]
                a_i = A[i, :]
                sv_j = Sv[Isv[j]] if Isv is not None and len(Isv) > j else Sv[j]
                sXv += a_i @ sv_j @ a_i.T
                if Av is not None and len(Av) > i:
                    s_j = S[:, j]
                    av_i = Av[i]
                    sXv += s_j.T @ av_i @ s_j + np.sum(sv_j * av_i)

        if Muv is not None and Muv.size > 0:
            sXv += np.sum(Muv[IX])

    # Determine whether to use priors based on Va
    use_prior = Va is not None and not np.any(np.isinf(Va))

    cost_x = 0.5 / V * sXv + 0.5 * ndata * np.log(2 * np.pi * V)
    cost_mu = 0.0
    cost_a = 0.0

    if use_prior:
        if Muv is not None and Muv.size > 0:
            cost_mu = 0.5 / Vmu * np.sum(Mu ** 2 + Muv) - 0.5 * np.sum(np.log(Muv)) + (n1 / 2) * np.log(Vmu) - (n1 / 2)
        elif Vmu != 0:
            cost_mu = 0.5 / Vmu * np.sum(Mu ** 2) + (n1 / 2) * np.log(2 * np.pi * Vmu)

        if Av is not None and len(Av) > 0:
            var1 = np.sum(np.sum(A ** 2, axis=0) / Va)
            var2 = (n1 / 2) * np.sum(np.log(Va), axis=0)
            var3 = (n1 * ncomp / 2)
            print("var 1 is ", var1)
            print("var 2 is ", var2)
            print("var 3 is ", var3)
            print("va is ", Va.shape)

            cost_a = 0.5 * np.sum(np.sum(A ** 2, axis=0) / Va) + (n1 / 2) * np.sum(np.log(Va), axis=0) - (n1 * ncomp / 2)
            print("cost_a after initial assignment:", cost_a)
            for i in range(n1):
                trace_term = 0.5 * np.sum(np.diag(Av[i]) / Va)
                sign, logdet = slogdet(Av[i])
                if sign > 0:
                    cost_a += trace_term - 0.5 * logdet
                else:
                    cost_a += trace_term - 0.5 * (-np.inf)
                print(f"cost_a after updating with trace and determinant of Av[{i}]:", cost_a)
        else:
            cost_a = 0.5 * np.sum(A ** 2 / Va) + (n1 / 2) * np.sum(np.log(2 * np.pi * Va))
            print("cost_a after assignment with Va:", cost_a)
    else:
        if Muv is not None and Muv.size > 0:
            cost_mu = -0.5 * np.sum(np.log(2 * np.pi * Muv)) - (n1 / 2)
        if Av is not None and len(Av) > 0:
            cost_a = - (n1 * ncomp / 2) * (1 + np.log(2 * np.pi))
            print("cost_a after no-prior initial assignment:", cost_a)
            for i in range(n1):
                sign, logdet = slogdet(Av[i])
                if sign > 0:
                    cost_a -= 0.5 * logdet
                else:
                    cost_a -= 0.5 * (-np.inf)
                print(f"cost_a after updating with determinant of Av[{i}]:", cost_a)

    cost_s = 0.5 * np.sum(S ** 2)
    if Sv is not None and len(Sv) > 0:
        if Isv is not None and len(Isv) > 0:
            for j in range(n2):
                sv_idx = Isv[j]
                if Sv[sv_idx] is not None:
                    trace_svj = 0.5 * np.trace(Sv[sv_idx])
                    sign, logdet_svj = slogdet(Sv[sv_idx])
                    if sign > 0:
                        cost_s += trace_svj - 0.5 * logdet_svj
                    else:
                        cost_s += trace_svj - 0.5 * (-np.inf)
        else:
            for j in range(n2):
                if Sv[j] is not None:
                    trace_svj = 0.5 * np.trace(Sv[j])
                    sign, logdet_svj = slogdet(Sv[j])
                    if sign > 0:
                        cost_s += trace_svj - 0.5 * logdet_svj
                    else:
                        cost_s += trace_svj - 0.5 * (-np.inf)
    cost_s -= (ncomp * n2) / 2
    cost = cost_mu + cost_a + cost_x + cost_s

    return cost, cost_x, cost_a, cost_mu, cost_s


In [72]:
# Test script for cf_full with representative data in Python

import numpy as np

# Initialize Av with the provided data
# Since actual 2x2 matrices are not fully provided, I'll use representative examples
# Adjust the size of Av to match the number of features (18 in your case)

num_features = 18  # Number of features (rows in A)
Av = []

for idx in range(num_features):
    # Replace with your actual 2x2 matrices if available
    Av.append(np.array([[0.0075, 0.0],
                        [0.0,    0.0075]]))

# Initialize X with representative data
# Using a subset of columns to keep the script manageable
X = np.array([
    [-0.3930,  0.6070, -0.3930,  0.6070,  0.6070],
    [-0.0148, -0.0148, -0.0148, -0.0148, -0.0148],
    [ 0.4078, -0.5922,  0.4078, -0.5922, -0.5922],
    [ 0.1267,  0.1267,  0.1267, -0.8733,  0.1267],
    [-0.1106, -0.1106, -0.1106,  0.8894, -0.1106],
    [-0.0161, -0.0161, -0.0161, -0.0161, -0.0161],
    [-0.0735, -0.0735, -0.0735, -0.0735, -0.0735],
    [-0.1472, -0.1472, -0.1472, -0.1472, -0.1472],
    [-0.4101,  0.5899, -0.4101,  0.5899,  0.5899],
    [ 0.6308, -0.3692,  0.6308, -0.3692, -0.3692],
    [ 0.5907, -0.4093, -0.4093,  0.5907, -0.4093],
    [-0.2348,  0.7652,  0.7652, -0.2348, -0.2348],
    [-0.1137, -0.1137, -0.1137, -0.1137, -0.1137],
    [-0.2422, -0.2422, -0.2422, -0.2422,  0.7578],
    [ 0.8113, -0.1887, -0.1887, -0.1887,  0.0],
    [-0.1889, -0.1889, -0.1889, -0.1889,  0.0],
    [-0.2492, -0.2492,  0.7508, -0.2492,  0.0],
    [-0.3733,  0.6267, -0.3733,  0.6267,  0.0]
])

# Initialize A with the provided data
A = np.array([
   [-0.4017,    0.0231],
   [-0.0000,    0.0112],
   [ 0.4017,   -0.0344],
   [ 0.1571,    0.0077],
   [-0.1329,    0.0001],
   [-0.0242,   -0.0078],
   [-0.0038,   -0.0133],
   [ 0.0027,   -0.0280],
   [-0.1788,   -0.2508],
   [ 0.1799,    0.2921],
   [ 0.0093,   -0.2902],
   [ 0.0113,    0.0227],
   [ 0.0474,    0.0429],
   [-0.0680,    0.2246],
   [ 0.1418,   -0.1016],
   [ 0.1203,   -0.0880],
   [ 0.0411,    0.0659],
   [-0.3032,    0.1236]
])

# Initialize S with representative data
S = np.array([
    [ 1.1116, -1.2595,  0.9658, -1.6907, -1.2819],
    [-0.2124, -0.0981,  0.8818, -0.8254,  0.3459]
])

# Initialize Mu with the provided data
Mu = np.array([
    0.0049,
   -0.0001,
   -0.0048,
   -0.0054,
    0.0045,
    0.0009,
   -0.0003,
   -0.0004,
   -0.0079,
    0.0086,
    0.0002,
   -0.0000,
    0.0000,
   -0.0002,
   -0.0007,
   -0.0005,
   -0.0008,
    0.0020
])

# Initialize V with the provided value
V = 0.1092

# Initialize Sv with representative data
num_samples = S.shape[1]  # Number of samples (columns in S)
Sv = []

for idx in range(num_samples):
    # Replace with your actual 2x2 matrices if available
    Sv.append(np.array([[0.0075, 0.0],
                        [0.0,    0.0075]]))

# Initialize Isv as empty as provided
Isv = []

# Initialize Muv with the provided data
Muv = np.array([
    0.8146e-3,
    0.8146e-3,
    0.8146e-3,
    0.8269e-3,
    0.8269e-3,
    0.8269e-3,
    0.8947e-3,
    0.8947e-3,
    0.8947e-3,
    0.8947e-3,
    0.8269e-3,
    0.8269e-3,
    0.8269e-3,
    0.8269e-3,
    0.8269e-3,
    0.8269e-3,
    0.8269e-3,
    0.8269e-3
])

# Initialize Va with the provided value
Va = np.array([1000.0, 1000.0])

# Initialize Vmu with the provided value
Vmu = 1000.0

# Initialize M with representative data
M = np.array([
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1],
    [1, 0, 1, 1, 1],
    [1, 0, 1, 1, 1],
    [1, 0, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1]
], dtype=bool)

# Initialize sXv with the provided value
sXv = 255.6381

# Initialize ndata with the provided value
ndata = 2342

# Now, call the cf_full function
# Ensure that the cf_full function is defined and accessible in your Python environment
# You need to have a Python version of cf_full implemented


cost, cost_x, cost_a, cost_mu, cost_s = cf_full(
    X, A, S, Mu, V, Av, Sv, Isv, Muv, Va, Vmu, M, sXv, ndata
)

# Display the results
print('Test Case Results:')
print(f'Total Cost: {cost}')
print(f'Cost_x: {cost_x}')
print(f'Cost_a: {cost_a}')
print(f'Cost_mu: {cost_mu}')
print(f'Cost_s: {cost_s}')


var 1 is  0.0008911428600000001
var 2 is  124.33959502167846
var 3 is  18.0
va is  (2,)
cost_a after initial assignment: 106.34004059310847
cost_a after updating with trace and determinant of Av[0]: 111.23290035154834
cost_a after updating with trace and determinant of Av[1]: 116.1257601099882
cost_a after updating with trace and determinant of Av[2]: 121.01861986842808
cost_a after updating with trace and determinant of Av[3]: 125.91147962686794
cost_a after updating with trace and determinant of Av[4]: 130.80433938530783
cost_a after updating with trace and determinant of Av[5]: 135.6971991437477
cost_a after updating with trace and determinant of Av[6]: 140.59005890218756
cost_a after updating with trace and determinant of Av[7]: 145.48291866062743
cost_a after updating with trace and determinant of Av[8]: 150.3757784190673
cost_a after updating with trace and determinant of Av[9]: 155.26863817750717
cost_a after updating with trace and determinant of Av[10]: 160.16149793594704
cost